# SEC-MALS Plotting Notebook (Google Colab)

Creates publication-quality SEC-MALS chromatograms from ASTRA exports.

**Input format:** Tab-separated ASTRA export with European number format (comma as decimal separator, period as thousands separator).

**Expected column layout:**
```
Columns 0–7 : BSA standard  →  (vol, UV), (vol, dRI), (vol, UV_dup), (vol, LS_raw)
Columns 8–15: Sample         →  (vol, UV), (vol, dRI), (vol, UV_dup), (vol, Mw_Da)
```
The Mw column contains data only inside the integration window set in ASTRA.

**How to use:**
1. Run **Cell 1** once to verify dependencies.
2. Run **Cell 2** to set global plot style.
3. Run **Cell 3** — a file picker will appear; upload your ASTRA `.txt` export.
4. Run subsequent cells to inspect and plot the data.
5. Run the last cell to save PDF/PNG and download them to your local machine.

In [ ]:
# ── Setup — run once ──────────────────────────────────────────────────────────
# numpy, pandas, and matplotlib are pre-installed in Colab.
# This cell installs any that may be missing (e.g. in a custom runtime).
import importlib, subprocess, sys

_required = ['numpy', 'pandas', 'matplotlib']
_missing  = [p for p in _required if importlib.util.find_spec(p) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + _missing)
    print(f'Installed: {_missing}')
else:
    print('All dependencies present:', _required)

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

# ── Global style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'pdf.fonttype': 42,   # editable text in Illustrator
    'svg.fonttype': 'none',
})

## 1. Upload data file

In [ ]:
# ── Upload your ASTRA export (.txt) ───────────────────────────────────────────
from google.colab import files as _colab_files

print('A file picker will appear below — select your ASTRA export:')
_uploaded = _colab_files.upload()

_fname    = next(iter(_uploaded))
DATA_FILE = Path(_fname)   # used for .stem/.name only

# ── Labels and options (edit here) ────────────────────────────────────────────
SAMPLE_LABEL = DATA_FILE.stem   # e.g. change to 'DynA'
BSA_LABEL    = 'BSA'
SHOW_BSA     = True             # set False if your file has no BSA reference

# Molar mass y-axis limits — shared by overview and publication plot
MW_YLIM      = (1, 10_000)     # kDa  (1 kDa – 10,000 kDa = 10 MDa, log scale)

print(f'\nFile  : {_fname}')
print(f'Label : {SAMPLE_LABEL!r}  (edit SAMPLE_LABEL above if needed)')
print(f'BSA   : {"shown" if SHOW_BSA else "hidden"}')


# ── Parser ────────────────────────────────────────────────────────────────────
def _eu(s: str) -> float:
    """Parse a European-format number ('1.234,56' → 1234.56)."""
    s = str(s).strip()
    if s in ('', 'nan', 'NaN', 'None'):
        return np.nan
    s = s.replace('.', '')   # strip thousands sep
    s = s.replace(',', '.')  # replace decimal comma
    try:
        return float(s)
    except ValueError:
        return np.nan


def load_astra_export(src) -> pd.DataFrame:
    """
    Load a two-sample ASTRA SEC-MALS export file.
    src may be a file path (str/Path) or a file-like object (e.g. io.StringIO).

    Column layout (pairs of volume, signal):
      0–1  : BSA UV        2–3  : BSA dRI
      4–5  : BSA UV dup    6–7  : BSA raw LS  (integration window only)
      8–9  : Sample UV     10–11: Sample dRI
      12–13: Sample UV dup 14–15: Sample Mw in Da  (integration window only)
    """
    col_names = [
        'bsa_uv_vol',  'bsa_uv',
        'bsa_ri_vol',  'bsa_ri',
        '_bsa_uv2v',   '_bsa_uv2',
        'bsa_ls_vol',  'bsa_ls',
        'dyna_uv_vol', 'dyna_uv',
        'dyna_ri_vol', 'dyna_ri',
        '_dyna_uv2v',  '_dyna_uv2',
        'dyna_mw_vol', 'dyna_mw_da',
    ]

    if hasattr(src, 'read'):
        fh, _close = src, False
    else:
        fh, _close = open(src, encoding='utf-8'), True

    rows = []
    try:
        for i, line in enumerate(fh):
            if i == 0:      # header row — skip
                continue
            parts = line.rstrip('\n').split('\t')
            vals = [_eu(p) for p in parts[0:16]]
            while len(vals) < 16:
                vals.append(np.nan)
            rows.append(vals)
    finally:
        if _close:
            fh.close()

    return pd.DataFrame(rows, columns=col_names)


df = load_astra_export(io.StringIO(_uploaded[_fname].decode('utf-8')))
print(f'\nLoaded {len(df):,} data rows.')

# ── Quick summary ──────────────────────────────────────────────────────────────
def _rng(col):
    s = df[col].dropna()
    if len(s) == 0:
        return 'no data'
    return f'{s.min():.3f} – {s.max():.3f}  (n={len(s):,})'

if SHOW_BSA:
    print(f"\nBSA  UV   volume : {_rng('bsa_uv_vol')} mL")
    print(f"BSA  UV   signal : {_rng('bsa_uv')}")
    print(f"BSA  dRI  volume : {_rng('bsa_ri_vol')} mL")
    print(f"BSA  dRI  signal : {_rng('bsa_ri')}")
    print()
print(f"Smpl UV   volume : {_rng('dyna_uv_vol')} mL")
print(f"Smpl UV   signal : {_rng('dyna_uv')}")
print(f"Smpl dRI  volume : {_rng('dyna_ri_vol')} mL")
print(f"Smpl dRI  signal : {_rng('dyna_ri')}")
print()
mw_data = df.dropna(subset=['dyna_mw_vol', 'dyna_mw_da'])
if len(mw_data):
    print(f"Smpl Mw   volume : {mw_data['dyna_mw_vol'].min():.2f} – {mw_data['dyna_mw_vol'].max():.2f} mL  (n={len(mw_data)})")
    print(f"Smpl Mw   range  : {mw_data['dyna_mw_da'].min()/1e3:.1f} – {mw_data['dyna_mw_da'].max()/1e3:.1f} kDa")
    print(f"Smpl Mw   median : {mw_data['dyna_mw_da'].median()/1e3:.1f} kDa")
else:
    print("Smpl Mw   : no data in integration window")

## 2. Quick overview — all signals

In [ ]:
# ── Layout: 2×2 with BSA, or 1×2 without ────────────────────────────────────
if SHOW_BSA:
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    pairs = [
        ('bsa_uv_vol',  'bsa_uv',  f'{BSA_LABEL} UV',    axes[0, 0], 'steelblue'),
        ('bsa_ri_vol',  'bsa_ri',  f'{BSA_LABEL} dRI',   axes[0, 1], 'forestgreen'),
        ('dyna_uv_vol', 'dyna_uv', f'{SAMPLE_LABEL} UV',  axes[1, 0], 'steelblue'),
        ('dyna_ri_vol', 'dyna_ri', f'{SAMPLE_LABEL} dRI', axes[1, 1], 'forestgreen'),
    ]
    ax_sample_uv = axes[1, 0]
else:
    fig, (ax_sample_uv, ax_ri_overview) = plt.subplots(1, 2, figsize=(10, 3.8))
    pairs = [
        ('dyna_uv_vol', 'dyna_uv', f'{SAMPLE_LABEL} UV',  ax_sample_uv,    'steelblue'),
        ('dyna_ri_vol', 'dyna_ri', f'{SAMPLE_LABEL} dRI', ax_ri_overview, 'forestgreen'),
    ]

for vcol, scol, title, ax, color in pairs:
    sub = df[[vcol, scol]].dropna()
    ax.plot(sub[vcol], sub[scol], color=color, lw=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Volume (mL)')
    ax.set_ylabel('Signal')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Mw overlay on sample UV panel (log scale)
ax_mw = ax_sample_uv.twinx()
if len(mw_data):
    ax_mw.scatter(mw_data['dyna_mw_vol'], mw_data['dyna_mw_da'] / 1e3,
                  color='firebrick', s=12, zorder=5, alpha=0.9)
ax_mw.set_yscale('log')
ax_mw.set_ylim(*MW_YLIM)
ax_mw.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
ax_mw.set_ylabel('Molar mass (kDa)', color='firebrick')
ax_mw.tick_params(axis='y', labelcolor='firebrick')

fig.suptitle(f'{SAMPLE_LABEL} — overview', fontsize=12)
fig.tight_layout()
plt.show()

## 3. Publication-quality SEC-MALS plot

In [ ]:
# ── Parameters — adjust to match your elution volumes ──────────────────────────
SAMPLE_VOL_RANGE = (8.0, 18.0)    # mL  — window around sample peak
BSA_VOL_RANGE    = (21.0, 29.0)   # mL  — window around BSA peak (None to hide)

# MW_YLIM is set in the upload cell above

# Only show Mw dots where dRI ≥ this fraction of peak dRI
MW_RI_THRESHOLD = 0.10

# Colors
C_MW     = '#f0c106'   # molar mass dots (left axis)
C_SAMPLE = '#1a4f8a'   # sample dRI trace (right axis)
C_BSA    = '#888888'   # BSA reference trace


# ── Helper ────────────────────────────────────────────────────────────────────
def _slice(df, vcol, scol, vmin, vmax):
    m = (df[vcol] >= vmin) & (df[vcol] <= vmax)
    return df.loc[m, vcol].values, df.loc[m, scol].values


# ── Layout ────────────────────────────────────────────────────────────────────
show_bsa = SHOW_BSA and BSA_VOL_RANGE is not None
if show_bsa:
    fig, (ax_d, ax_b) = plt.subplots(1, 2, figsize=(9, 3.8),
                                      gridspec_kw={'wspace': 0.55})
else:
    fig, ax_d = plt.subplots(1, 1, figsize=(4.5, 3.8))


# ── Sample panel: left = Mw (log), right = dRI ────────────────────────────────
ax_ri = ax_d.twinx()

vd, sd = _slice(df, 'dyna_ri_vol', 'dyna_ri', *SAMPLE_VOL_RANGE)
sd_norm = sd / sd.max() if sd.max() > 0 else sd
ax_ri.plot(vd, sd_norm, color=C_SAMPLE, lw=1.5, label=SAMPLE_LABEL)
ax_ri.set_ylim(-0.06, 1.05)
ax_ri.set_ylabel('Normalized dRI', color='#000000', fontsize=11)
ax_ri.tick_params(axis='y', labelcolor='#000000')

mw_sub = mw_data[
    (mw_data['dyna_mw_vol'] >= SAMPLE_VOL_RANGE[0]) &
    (mw_data['dyna_mw_vol'] <= SAMPLE_VOL_RANGE[1])
].copy()
if len(mw_sub) > 0 and len(vd) > 0:
    ri_at_mw = np.interp(mw_sub['dyna_mw_vol'], vd, sd_norm)
    mw_sub = mw_sub[ri_at_mw >= MW_RI_THRESHOLD]

mw_kda = mw_sub['dyna_mw_da'].values / 1e3
vol_mw = mw_sub['dyna_mw_vol'].values

ax_d.scatter(vol_mw, mw_kda, color=C_MW, s=4, zorder=5, alpha=0.8, label='Molar mass')

if len(mw_kda) > 0:
    mw_median = float(np.median(mw_kda))
    ax_d.axhline(mw_median, color=C_MW, lw=0.8, ls='--', alpha=0.5)
    ax_d.annotate(
        f'{mw_median:.0f} kDa',
        xy=(vol_mw.max(), mw_median),
        xytext=(5, 2), textcoords='offset points',
        color=C_MW, fontsize=9, va='center', ha='left',
    )

ax_d.set_xlim(SAMPLE_VOL_RANGE)
ax_d.set_ylim(*MW_YLIM)
ax_d.set_yscale('log')
ax_d.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
ax_d.set_xlabel('Elution volume (mL)', fontsize=11)
ax_d.set_ylabel('Molar mass (kDa)', color='#000000', fontsize=11)
ax_d.tick_params(axis='y', labelcolor='#000000')
ax_d.set_title(SAMPLE_LABEL, fontsize=11, fontweight='bold')

for ax in (ax_d, ax_ri):
    for spine in ax.spines.values():
        spine.set_visible(True)

h1, l1 = ax_ri.get_legend_handles_labels()
h2, l2 = ax_d.get_legend_handles_labels()
ax_d.legend(h1 + h2, l1 + l2, loc='upper right', frameon=False, fontsize=9)


# ── BSA panel ─────────────────────────────────────────────────────────────────
if show_bsa:
    vb, sb = _slice(df, 'bsa_ri_vol', 'bsa_ri', *BSA_VOL_RANGE)
    sb_norm = sb / sb.max() if (len(sb) > 0 and sb.max() > 0) else sb

    ax_b.plot(vb, sb_norm, color=C_BSA, lw=1.5, label=BSA_LABEL)
    ax_b.set_xlim(BSA_VOL_RANGE)
    ax_b.set_ylim(-0.06, 1.05)
    ax_b.set_xlabel('Elution volume (mL)', fontsize=11)
    ax_b.set_ylabel('Normalized dRI', fontsize=11)
    ax_b.set_title(BSA_LABEL, fontsize=11, fontweight='bold')
    for spine in ax_b.spines.values():
        spine.set_visible(True)
    ax_b.legend(loc='upper right', frameon=False, fontsize=9)

fig.tight_layout()
plt.show()

## 4. Save figure and download

In [ ]:
# ── Save to Colab storage and download to your local machine ───────────────────
from google.colab import files as _colab_files

OUTPUT_STEM = DATA_FILE.stem   # e.g. '20251217_dynA'

_outputs = []
for ext in ('pdf', 'png'):
    out = f'/content/{OUTPUT_STEM}_sec_mals.{ext}'
    fig.savefig(out, dpi=300, bbox_inches='tight')
    print(f'Saved → {out}')
    _outputs.append(out)

print('\nStarting downloads...')
for out in _outputs:
    _colab_files.download(out)